# StructurePreprocessor v1.1 — end-to-end demo

Run AF model + PAE features through `CPP.run_num` against a small synthetic AlphaFold-style fixture (`AF_TINY`). The fixture ships in `aaanalysis/_data/pdb_test/` so this notebook is self-contained and runs without any external downloads.

Each encoder normalizes its output to `[0, 1]` per the recipes documented in `StructurePreprocessor`'s class docstring.


In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import aaanalysis as aa

aa.options['verbose'] = False
warnings.filterwarnings('ignore')

PDB_FIXTURES = Path(aa.__file__).resolve().parent / '_data' / 'pdb_test'
stp = aa.StructurePreprocessor(verbose=False)
df_seq = pd.DataFrame({
    'entry':     ['AF_TINY'],
    'sequence':  ['ACDEFGHIKLMNPQRSTVWYACDEFGHIKL'],
})
df_seq

## 1. AF model-file features via `encode_pdb`

Reads pLDDT from the B-factor column of `AF_TINY.pdb`; computes side-chain chi1 and CA-CA contact density; outputs normalized to `[0, 1]`.

In [ ]:
pdb_feats = ['plddt', 'plddt_disorder', 'plddt_tier',
             'chi1_sincos', 'ca_centroid_dist_norm', 'contact_count_8A']
dict_pdb, df_seq_pdb = stp.encode_pdb(
    df_seq=df_seq, pdb_folder=str(PDB_FIXTURES),
    features=pdb_feats)
print('dict_pdb shape per entry:', {k: v.shape for k, v in dict_pdb.items()})
print('value range:', float(np.nanmin(dict_pdb['AF_TINY'])),
      float(np.nanmax(dict_pdb['AF_TINY'])))

## 2. AF PAE sidecar features via `encode_pae`

Loads `AF_TINY_pae.json` (the AF predicted-aligned-error matrix); summarizes per residue as row-mean / local-mean / distal-mean / asymmetry. All normalized to `[0, 1]` (divisor 31.75 Å for most keys, 10 Å for `pae_asymmetry`).

In [ ]:
import tempfile, shutil
pae_dir = tempfile.mkdtemp()
shutil.copy(PDB_FIXTURES / 'AF_TINY_pae.json', Path(pae_dir) / 'AF_TINY.json')

pae_feats = ['pae_row_mean', 'pae_local_mean', 'pae_distal_mean',
             'pae_asymmetry']
dict_pae, df_seq_pae = stp.encode_pae(
    df_seq=df_seq, pae_folder=pae_dir,
    features=pae_feats, local_window=5)
print('dict_pae shape per entry:', {k: v.shape for k, v in dict_pae.items()})
print('local vs distal mean PAE on AF_TINY:',
      float(np.nanmean(dict_pae['AF_TINY'][:, 1])), 'vs',
      float(np.nanmean(dict_pae['AF_TINY'][:, 2])))

## 3. Combine + build (`df_scales` + `df_cat`)

`combine_dict_nums` stitches the two `dict_num`s along the D axis. `build_pseudo_scales` populates the per-AA-averaged df_scales from the user corpus — needed for the redundancy filter's `max_cor` arm to be meaningful. `build_cat` produces the (D, 5) metadata.

In [ ]:
feats = pdb_feats + pae_feats
dict_num = aa.combine_dict_nums(dict_nums=[dict_pdb, dict_pae])
df_scales = stp.build_pseudo_scales(
    df_seq=df_seq, dict_num=dict_num, features=feats)
df_cat = stp.build_cat(features=feats)
print('df_scales:', df_scales.shape, ' df_cat:', df_cat.shape)
df_cat.head(8)

Every category resolves to the locked `Structure` color (`#2E6E5E`) — that closes the v1 CPPPlot defect where unknown categories raised `ValueError`.

In [ ]:
import aaanalysis.utils as ut
print('unique categories:', df_cat['category'].unique().tolist())
print('color for Structure:', ut.DICT_COLOR_CAT.get('Structure'))

## What next

With a real corpus this `(df_scales, df_cat, dict_num)` triple plugs into `NumericalFeature.get_parts(...)` and `CPP.run_num(...)` exactly the way the integration test in `tests/unit/cpp_tests/test_run_num_structural.py` does. AF-DB bulk downloads can use the `<entry>.cif.gz` resolver path; PAE sidecars can use the AF-DB canonical filename `AF-<entry>-F1-predicted_aligned_error_v4.json` directly without renaming.